In [1]:
from pyspark.sql import functions as F

df_silver_traffic = (
    spark.read
        .format("delta")
        .load("abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Silver.Lakehouse/Tables/dbo/fact_seo_traffic")
)

print("Silver traffic rows:", df_silver_traffic.count())
print("Silver traffic columns:", len(df_silver_traffic.columns))

display(df_silver_traffic.limit(10))

StatementMeta(, fa960f9c-079d-4034-a996-32f0c467e51e, 3, Finished, Available, Finished, False)

Silver traffic rows: 49995
Silver traffic columns: 14


SynapseWidget(Synapse.DataFrame, 48eb29cd-c197-438b-abf4-7e45b5088d8f)

In [3]:
from pyspark.sql import functions as F

df_date = spark.read.table("dim_date")
df_country = spark.read.table("dim_country")
df_device = spark.read.table("dim_device")
df_product = spark.read.table("dim_product")
df_page = spark.read.table("dim_page")
df_keyword = spark.read.table("dim_keyword")

StatementMeta(, fa960f9c-079d-4034-a996-32f0c467e51e, 5, Finished, Available, Finished, False)

In [4]:
print("Date:", df_date.count())
print("Country:", df_country.count())
print("Device:", df_device.count())
print("Product:", df_product.count())
print("Page:", df_page.count())
print("Keyword:", df_keyword.count())

StatementMeta(, fa960f9c-079d-4034-a996-32f0c467e51e, 6, Finished, Available, Finished, False)

Date: 365
Country: 10
Device: 3
Product: 50
Page: 500
Keyword: 2000


In [5]:
df_gold_traffic = (
    df_silver_traffic.alias("f")

    # Date
    .join(
        df_date.select("date_key", "date").alias("d"),
        F.col("f.event_date") == F.col("d.date"),
        "left"
    )

    # Country
    .join(
        df_country.select("country_key", "country_code").alias("c"),
        F.col("f.country_code") == F.col("c.country_code"),
        "left"
    )

    # Device
    .join(
        df_device.select("device_key", "device_type").alias("dv"),
        F.col("f.device_type") == F.col("dv.device_type"),
        "left"
    )

    # Product
    .join(
        df_product.select("product_key", "product_id").alias("p"),
        F.col("f.product_id") == F.col("p.product_id"),
        "left"
    )

    # Page
    .join(
        df_page.select("page_key", "page_id").alias("pg"),
        F.col("f.page_id") == F.col("pg.page_id"),
        "left"
    )

    # Keyword
    .join(
        df_keyword.select("keyword_key", "keyword_id").alias("k"),
        F.col("f.keyword_id") == F.col("k.keyword_id"),
        "left"
    )
)

StatementMeta(, fa960f9c-079d-4034-a996-32f0c467e51e, 7, Finished, Available, Finished, False)

In [6]:
df_gold_traffic = df_gold_traffic.select(
    F.col("f.traffic_id"),
    F.col("d.date_key"),
    F.col("c.country_key"),
    F.col("dv.device_key"),
    F.col("p.product_key"),
    F.col("pg.page_key"),
    F.col("k.keyword_key"),
    F.col("f.visits"),
    F.col("f.clicks"),
    F.col("f.impressions"),
    F.col("f.orders"),
    F.col("f.revenue"),
    F.col("f.traffic_type"),
    F.col("f.updated_timestamp")
)

StatementMeta(, fa960f9c-079d-4034-a996-32f0c467e51e, 8, Finished, Available, Finished, False)

In [7]:
print("Gold traffic rows:", df_gold_traffic.count())
print("Gold traffic columns:", len(df_gold_traffic.columns))

display(df_gold_traffic.limit(10))

StatementMeta(, fa960f9c-079d-4034-a996-32f0c467e51e, 9, Finished, Available, Finished, False)

Gold traffic rows: 49995
Gold traffic columns: 14


SynapseWidget(Synapse.DataFrame, 64c94c23-f816-4d87-93fc-aca3a33c6a97)

In [8]:
id_columns = [
    "date_key",
    "country_key",
    "device_key",
    "product_key",
    "page_key",
    "keyword_key"
]

for col_name in id_columns:
    null_count = (
        df_gold_traffic
        .filter(F.col(col_name).isNull())
        .count()
    )

    print(f"{col_name}: {null_count} nulls")

StatementMeta(, fa960f9c-079d-4034-a996-32f0c467e51e, 10, Finished, Available, Finished, False)

date_key: 0 nulls
country_key: 0 nulls
device_key: 0 nulls
product_key: 5 nulls
page_key: 5 nulls
keyword_key: 0 nulls


In [9]:
display(
    df_gold_traffic
    .filter(
        F.col("product_key").isNull() |
        F.col("page_key").isNull()
    )
    .select(
        "traffic_id",
        "date_key",
        "country_key",
        "device_key",
        "product_key",
        "page_key",
        "keyword_key",
        "visits",
        "orders",
        "revenue"
    )
)

StatementMeta(, fa960f9c-079d-4034-a996-32f0c467e51e, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c5668ec6-9861-4f96-b06c-e4adac2e44e2)

In [10]:
display(
    df_silver_traffic
    .filter(
        F.col("product_id").isNull() |
        F.col("page_id").isNull()
    )
    .select(
        "traffic_id",
        "product_id",
        "page_id",
        "keyword_id",
        "visits",
        "orders",
        "revenue"
    )
)

StatementMeta(, fa960f9c-079d-4034-a996-32f0c467e51e, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3399f379-7317-48cd-885f-78d49daabb8b)

In [11]:
print("Source Product IDs null:",
      df_silver_traffic.filter(F.col("product_id").isNull()).count())

print("Gold Product Keys null:",
      df_gold_traffic.filter(F.col("product_key").isNull()).count())

print("Source Page IDs null:",
      df_silver_traffic.filter(F.col("page_id").isNull()).count())

print("Gold Page Keys null:",
      df_gold_traffic.filter(F.col("page_key").isNull()).count())

StatementMeta(, fa960f9c-079d-4034-a996-32f0c467e51e, 13, Finished, Available, Finished, False)

Source Product IDs null: 5
Gold Product Keys null: 5
Source Page IDs null: 5
Gold Page Keys null: 5


In [12]:
(
    df_gold_traffic.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("fact_seo_traffic")
)

StatementMeta(, fa960f9c-079d-4034-a996-32f0c467e51e, 14, Finished, Available, Finished, False)

In [13]:
df_gold_traffic_check = spark.read.table("fact_seo_traffic")

print(
    "Gold fact_seo_traffic rows:",
    df_gold_traffic_check.count()
)

print(
    "Gold fact_seo_traffic columns:",
    len(df_gold_traffic_check.columns)
)

print(
    "Duplicate traffic IDs:",
    df_gold_traffic_check
        .groupBy("traffic_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

display(
    df_gold_traffic_check.limit(10)
)

StatementMeta(, fa960f9c-079d-4034-a996-32f0c467e51e, 15, Finished, Available, Finished, False)

Gold fact_seo_traffic rows: 49995
Gold fact_seo_traffic columns: 14
Duplicate traffic IDs: 0


SynapseWidget(Synapse.DataFrame, b4c92412-44f4-457c-b6d9-42e53e31b992)